## HuggingFace Pool: NumPy Random Tensor

A compact end-to-end example that creates a random NumPy tensor, uploads it to Hugging Face Hub with `HuggingFacePool`, retrieves it back, and validates integrity.

### Prerequisites
- `huggingface_hub` installed
- HF dataset or model repo created
- valid HF token with write/read access

### Credentials
This notebook reads values from:
`laila.args`

Expected keys:
- `HF_REPO_ID` (e.g. `amirzadeh/test`)
- `HF_REPO_TYPE` (`dataset` or `model`, default `dataset`)
- `HF_TOKEN`
- `HF_REVISION` (optional, default `main`)


In [ ]:
import numpy as np
import laila


In [ ]:
from laila.pool import HuggingFacePool

# These argument names are arbitrary and can be changed by the user.
# laila.args is just a container for user-provided arguments.
# Replace these placeholders with your own Hugging Face values before running.
laila.args.HF_REPO_ID = "your-username/your-repo"
laila.args.HF_REPO_TYPE = "dataset"
laila.args.HF_TOKEN = "your-huggingface-token"
laila.args.HF_REVISION = "main"

hf_pool = HuggingFacePool(
    repo_id=laila.args.HF_REPO_ID,
    repo_type=laila.args.HF_REPO_TYPE,
    revision=laila.args.HF_REVISION,
    token=laila.args.HF_TOKEN,
)


In [ ]:
laila.memory.add_pool(hf_pool, pool_nickname="hf_pool")


In [ ]:
# Create a random NumPy tensor (array)
np.random.seed(42)
random_tensor = np.random.randn(256, 256).astype(np.float32)
wrapped_tensor = laila.constant(data=random_tensor)

print("Tensor shape:", random_tensor.shape)
print("Tensor dtype:", random_tensor.dtype)
print("Global ID:", wrapped_tensor.global_id)


In [ ]:
# Upload to Hugging Face pool
future_memorize = laila.memorize(wrapped_tensor, pool_nickname="hf_pool")
print("Before wait:", laila.status(future_memorize))
laila.wait(future_memorize)
print("After wait:", laila.status(future_memorize))


In [ ]:
# Download from Hugging Face pool
future_remember = laila.remember(wrapped_tensor.global_id, pool_nickname="hf_pool")
recovered = laila.wait(future_remember)

assert np.array_equal(recovered.data, random_tensor)
print("Recovery verified: arrays are equal")


In [ ]:
# Optional cleanup
future_forget = laila.forget(wrapped_tensor.global_id, pool_nickname="hf_pool")
laila.wait(future_forget)
print("Cleanup complete")
